In [ ]:
import torch
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import random_split,DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn 
import torch.optim as optim

In [ ]:
transform = transforms.Compose([
                                  transforms.RandomHorizontalFlip(),
                                  transforms.RandomRotation(15),
                                  transforms.ColorJitter(brightness=0.2,contrast=0.2, saturation=0.2),
                                  transforms.Resize((224, 224)),
                                  transforms.ToTensor(),
                                  transforms.Normalize(mean=[0.5,0.5,0.5],std=[0.5,0.5,0.5])
                                ])
dataset = ImageFolder(root="FRUIT-16K", transform=transform)

In [ ]:
dataset

In [ ]:
classes=dataset.classes
classes

In [ ]:
num_classes=len(classes)
num_classes

In [ ]:
train_size=int(0.75*len(dataset))
val_size=len(dataset)-train_size

train_size,val_size

In [ ]:
train_dataset,val_dataset=random_split(dataset,[train_size,val_size])

In [ ]:

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=True)

In [ ]:
for images,labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

In [ ]:
images[0].shape

In [ ]:
images[0].permute(1,2,0).shape

In [ ]:
classes[labels[10]]

In [ ]:
plt.imshow(images[10].permute(1,2,0))
plt.show()

# Model Training

In [ ]:
class Freshness_inspection_1(nn.Module):
  def __init__(self):
    super().__init__()
    self.network = nn.Sequential(
                                    nn.Conv2d(3,32,kernel_size=(3,3),padding='same'),  # output (32,224,224)
                                    nn.ReLU(),
                                    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),   # output (32,112,112)
                                    nn.Conv2d(32,64,kernel_size=(3,3)),  # output (64,110,110)
                                    nn.ReLU(),
                                    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),   # output (64,55,55)
                                    
                                    nn.Flatten(),
                                    nn.Linear(64*55*55, 1000),
                                    nn.ReLU(),
                                    nn.Linear(1000,16)
                                )
  def forward(self,x):
    return self.network(x)

In [ ]:
device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

In [ ]:
import time
def train_model(model, criterion, optimizer, epochs=5):
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            
            if (batch_num+1) % 10 == 0:
                print(f"Batch: {batch_num+1}, Epoch: {epoch+1}, Loss: {loss.item():0.2f}")
            
            running_loss += loss.item() * images.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Avg Loss: {epoch_loss:.4f}")

        # Validation
        model.eval()
        correct = 0
        total = 0
        all_labels = []
        all_predictions = []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data,1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())
                
            print(f"*** Validation Accuracy: {100 * correct / total:.2f}% ***")
            
    end = time.time()
    print(f"Execution time: {end - start} seconds")     
    
    return all_labels, all_predictions

In [ ]:
model = Freshness_inspection_1().to(device)
epochs = 10
learning_rate = 0.001
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=learning_rate)

all_labels, all_predictions = train_model(model, loss_fn, optimizer, epochs=10)

In [ ]:
import torch
import gc

del model

gc.collect()
torch.cuda.empty_cache()

print(
    "Free GPU memory:",
    torch.cuda.mem_get_info()[0] / 1024**3,
    "GB"
)

In [ ]:
 
import time
import torch
import matplotlib.pyplot as plt

def train_model(model, criterion, optimizer, epochs=5):

    start = time.time()

    # Store accuracy for each epoch
    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):

        # ---------------- Training ----------------
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_num, (images, labels) in enumerate(train_loader):

            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Training accuracy
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            if (batch_num + 1) % 10 == 0:
                print(f"Batch: {batch_num+1}, Epoch: {epoch+1}, Loss: {loss.item():0.2f}")

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        train_accuracy = 100 * train_correct / train_total
        train_accuracies.append(train_accuracy)

        print(f"Epoch [{epoch+1}/{epochs}], Avg Loss: {epoch_loss:.4f}")
        print(f"*** Training Accuracy: {train_accuracy:.2f}% ***")

        # ---------------- Validation ----------------
        model.eval()
        correct = 0
        total = 0

        all_labels = []
        all_predictions = []

        with torch.no_grad():

            for images, labels in val_loader:

                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())

        val_accuracy = 100 * correct / total
        val_accuracies.append(val_accuracy)

        print(f"*** Validation Accuracy: {val_accuracy:.2f}% ***")

    end = time.time()

    print(f"Execution time: {end - start:.2f} seconds")

    # ---------------- Plot Accuracy ----------------
    plt.figure(figsize=(8, 5))

    plt.plot(range(1, epochs+1), train_accuracies,
             marker='o', label='Training Accuracy')

    plt.plot(range(1, epochs+1), val_accuracies,
             marker='o', label='Validation Accuracy')

    plt.title("Training vs Validation Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.xticks(range(1, epochs+1))
    plt.legend()
    plt.grid(True)
    plt.show()

    return all_labels, all_predictions


In [ ]:
all_labels, all_predictions = train_model(model, loss_fn, optimizer)

# Batch Normalization

In [ ]:
class Freshness_CNN_BN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding='same'),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.AdaptiveAvgPool2d((1, 1)),

            nn.Flatten(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, 16)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
model = Freshness_CNN_BN().to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

all_labels, all_predictions = train_model(
    model, loss_fn, optimizer, epochs=10
)

# Batch Normalization + Dropout

In [ ]:
class Freshness_CNN_BN_Dropout(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding='same'),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1),

            nn.AdaptiveAvgPool2d((1, 1)),

            nn.Flatten(),

            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 16)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
model = Freshness_CNN_BN_Dropout().to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

all_labels, all_predictions = train_model(
    model, loss_fn, optimizer, epochs=10
)

# Batch Normalization + Dropout + Weight Decay

In [ ]:
model = Freshness_CNN_BN_Dropout().to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

all_labels, all_predictions = train_model(
    model, loss_fn, optimizer, epochs=10
)

# All techniques + Early Stopping

In [ ]:
model = Freshness_CNN_BN_Dropout().to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [ ]:

def train_model_early_stopping(
    model,
    criterion,
    optimizer,
    epochs=30,
    patience=5
):

    start = time.time()

    train_losses = []
    val_losses = []

    train_accuracies = []
    val_accuracies = []

    best_val_loss = float('inf')
    best_model_weights = copy.deepcopy(model.state_dict())

    patience_counter = 0

    best_labels = []
    best_predictions = []

    for epoch in range(epochs):

        # -------- Training --------
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / total
        train_accuracy = 100 * correct / total

        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)

        # -------- Validation --------
        model.eval()

        val_running_loss = 0.0
        correct = 0
        total = 0

        all_labels = []
        all_predictions = []

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())

        val_loss = val_running_loss / total
        val_accuracy = 100 * correct / total

        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)

        print(f"\nEpoch [{epoch+1}/{epochs}]")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Accuracy: {train_accuracy:.2f}%")
        print(f"Validation Loss: {val_loss:.4f}")
        print(f"Validation Accuracy: {val_accuracy:.2f}%")

        # -------- Early Stopping --------
        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_weights = copy.deepcopy(
                model.state_dict()
            )

            best_labels = all_labels.copy()
            best_predictions = all_predictions.copy()

            patience_counter = 0

            print("Best model saved!")

        else:

            patience_counter += 1

            print(
                f"No improvement: "
                f"{patience_counter}/{patience}"
            )

            if patience_counter >= patience:

                print("Early stopping triggered!")
                break

    # Restore best weights
    model.load_state_dict(best_model_weights)

    # -------- Plot Accuracy --------
    epochs_range = range(1, len(train_accuracies) + 1)

    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs_range,
        train_accuracies,
        marker='o',
        label='Training Accuracy'
    )

    plt.plot(
        epochs_range,
        val_accuracies,
        marker='o',
        label='Validation Accuracy'
    )

    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    # -------- Plot Loss --------
    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs_range,
        train_losses,
        marker='o',
        label='Training Loss'
    )

    plt.plot(
        epochs_range,
        val_losses,
        marker='o',
        label='Validation Loss'
    )

    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Execution Time: {time.time() - start:.2f} seconds")

    return best_labels, best_predictions


In [ ]:
model = Freshness_CNN_BN_Dropout().to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

all_labels, all_predictions = train_model_early_stopping(
    model,
    loss_fn,
    optimizer,
    epochs=30,
    patience=5
)